# 04 — SQL Analysis

At least 10 business-question queries: aggregations, JOINs, subqueries, GROUP BY / HAVING.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_DIR = '/content/drive/MyDrive/NHL_Analytics_Hub'
sys.path.append(PROJECT_DIR)
import nhl_api_utils as nhl
import pandas as pd

DB_PATH = f'{PROJECT_DIR}/data/nhl_hockey.db'
conn = nhl.get_connection(DB_PATH)


Mounted at /content/drive


In [ ]:
# ============================================================
# PHASE 4 — SQL ANALYTICS
# TASK 1 — SCHEMA INSPECTION
# ============================================================

tables = [
    "teams",
    "standings",
    "players",
    "games",
    "game_stats",
    "skater_season_stats",
    "goalie_season_stats"
]

for table in tables:
    print("=" * 70)
    print(f"TABLE: {table}")
    print("=" * 70)

    schema_df = pd.read_sql(
        f"PRAGMA table_info({table});",
        conn
    )

    print(schema_df.to_string(index=False))
    print()

TABLE: teams
 cid            name    type  notnull dflt_value  pk
   0         team_id INTEGER        0       None   1
   1     team_abbrev    TEXT        1       None   0
   2       team_name    TEXT        1       None   0
   3 conference_name    TEXT        0       None   0
   4   division_name    TEXT        0       None   0
   5        logo_url    TEXT        0       None   0

TABLE: standings
 cid          name    type  notnull dflt_value  pk
   0   standing_id INTEGER        0       None   1
   1       team_id INTEGER        1       None   0
   2        season    TEXT        0       None   0
   3  games_played INTEGER        0       None   0
   4          wins INTEGER        0       None   0
   5        losses INTEGER        0       None   0
   6     ot_losses INTEGER        0       None   0
   7        points INTEGER        0       None   0
   8     goals_for INTEGER        0       None   0
   9 goals_against INTEGER        0       None   0
  10     home_wins INTEGER        0  

In [ ]:
# ============================================================
# PHASE 4 — TASK 2
# Q1: TOP 10 TEAMS BY GOALS SCORED
# ============================================================

query = """
SELECT
    t.team_name,
    s.goals_for
FROM standings s
JOIN teams t
    ON t.team_id = s.team_id
ORDER BY s.goals_for DESC
LIMIT 10;
"""

result = pd.read_sql(query, conn)

result

,team_name,goals_for
0,Colorado Avalanche,302
1,Carolina Hurricanes,296
2,Pittsburgh Penguins,293
3,Tampa Bay Lightning,290
4,Buffalo Sabres,288
5,Montréal Canadiens,283
6,Edmonton Oilers,282
7,Dallas Stars,279
8,Ottawa Senators,278
9,Anaheim Ducks,273


### 1. Which team has scored the most total goals this season? (aggregation + ORDER BY)

In [ ]:
pd.read_sql('''
SELECT t.team_name, s.goals_for
FROM standings s JOIN teams t ON t.team_id = s.team_id
ORDER BY s.goals_for DESC
LIMIT 10;
''', conn)


### 2. Top 5 point scorers across the league (JOIN + ORDER BY)

In [ ]:
pd.read_sql('''
SELECT p.first_name, p.last_name, t.team_name, ss.points
FROM skater_season_stats ss
JOIN players p ON p.player_id = ss.player_id
JOIN teams t ON t.team_id = ss.team_id
ORDER BY ss.points DESC
LIMIT 5;
''', conn)


,first_name,last_name,team_name,points
0,Connor,McDavid,Edmonton Oilers,138
1,Nikita,Kucherov,Tampa Bay Lightning,130
2,Nathan,MacKinnon,Colorado Avalanche,127
3,Macklin,Celebrini,San Jose Sharks,115
4,Mark,Scheifele,Winnipeg Jets,103


### 3. Players with more than 20 goals AND more than 30 assists (WHERE, multiple conditions)

In [ ]:
pd.read_sql('''
SELECT p.first_name, p.last_name, ss.goals, ss.assists
FROM skater_season_stats ss
JOIN players p ON p.player_id = ss.player_id
WHERE ss.goals > 20 AND ss.assists > 30
ORDER BY ss.goals DESC;
''', conn)


,first_name,last_name,goals,assists
0,Nathan,MacKinnon,53,74
1,Cole,Caufield,51,37
2,Connor,McDavid,48,90
3,Macklin,Celebrini,45,70
4,Jason,Robertson,45,51
...,...,...,...,...
72,Jesper,Bratt,22,49
73,Darren,Raddysh,22,48
74,Brady,Tkachuk,22,37
75,Jackson,Blake,22,31


### 4. Teams with a points total above the league average (subquery)

In [ ]:
pd.read_sql('''
SELECT t.team_name, s.points
FROM standings s JOIN teams t ON t.team_id = s.team_id
WHERE s.points > (SELECT AVG(points) FROM standings)
ORDER BY s.points DESC;
''', conn)


,team_name,points
0,Colorado Avalanche,121
1,Carolina Hurricanes,113
2,Dallas Stars,112
3,Buffalo Sabres,109
4,Tampa Bay Lightning,106
5,Montréal Canadiens,106
6,Minnesota Wild,104
7,Boston Bruins,100
8,Ottawa Senators,99
9,Pittsburgh Penguins,98


### 5. Divisions with an average team points total above 90 (GROUP BY + HAVING)

In [ ]:
pd.read_sql('''
SELECT t.division_name, AVG(s.points) AS avg_points
FROM standings s JOIN teams t ON t.team_id = s.team_id
GROUP BY t.division_name
HAVING AVG(s.points) > 90
ORDER BY avg_points DESC;
''', conn)


,division_name,avg_points
0,Atlantic,96.750
1,Central,94.375
2,Metropolitan,93.875


### 6. Best save percentage among goalies with at least 10 games played

In [ ]:
pd.read_sql('''
SELECT p.first_name, p.last_name, gs.save_pct, gs.games_played
FROM goalie_season_stats gs
JOIN players p ON p.player_id = gs.player_id
WHERE gs.games_played >= 10
ORDER BY gs.save_pct DESC
LIMIT 10;
''', conn)


,first_name,last_name,save_pct,games_played
0,Scott,Wedgewood,0.92131,45
1,Jesper,Wallstedt,0.91568,35
2,Dennis,Hildeby,0.91423,20
3,Andrei,Vasilevskiy,0.91233,58
4,Logan,Thompson,0.91178,58
5,Igor,Shesterkin,0.91157,51
6,Ukko-Pekka,Luukkonen,0.90967,35
7,Anton,Forsberg,0.90966,36
8,Joel,Hofer,0.90960,46
9,Devin,Cooley,0.90909,31


### 7. Most penalty minutes, league-wide (aggregation + ORDER BY)

In [ ]:
pd.read_sql('''
SELECT p.first_name, p.last_name, ss.penalty_min
FROM skater_season_stats ss
JOIN players p ON p.player_id = ss.player_id
ORDER BY ss.penalty_min DESC
LIMIT 10;
''', conn)


,first_name,last_name,penalty_min
0,Nikita,Zadorov,152
1,Mark,Kastelic,140
2,Logan,Stanley,128
3,Tom,Wilson,117
4,Arber,Xhekaj,116
5,A.J.,Greer,113
6,Charle-Edouard,D'Astous,112
7,Adam,Klapka,112
8,Curtis,Douglas,108
9,Ross,Johnston,107


### 8. Home vs away win split per team (derived aggregation)

In [ ]:
pd.read_sql('''
SELECT t.team_name, s.home_wins, s.away_wins,
       (s.home_wins - s.away_wins) AS home_minus_away
FROM standings s JOIN teams t ON t.team_id = s.team_id
ORDER BY home_minus_away DESC;
''', conn)


,team_name,home_wins,away_wins,home_minus_away
0,Boston Bruins,29,16,13
1,Calgary Flames,23,11,12
2,Washington Capitals,25,18,7
3,Florida Panthers,23,17,6
4,Carolina Hurricanes,29,24,5
5,Anaheim Ducks,24,19,5
6,Nashville Predators,21,17,4
7,Seattle Kraken,19,15,4
8,Toronto Maple Leafs,18,14,4
9,Edmonton Oilers,22,19,3


### 9. Average points per game, per position (GROUP BY on players)

In [ ]:
pd.read_sql('''
SELECT p.position, ROUND(AVG(ss.points * 1.0 / NULLIF(ss.games_played,0)), 2) AS avg_points_per_game
FROM skater_season_stats ss
JOIN players p ON p.player_id = ss.player_id
WHERE ss.games_played > 0
GROUP BY p.position
ORDER BY avg_points_per_game DESC;
''', conn)


,position,avg_points_per_game
0,R,0.47
1,C,0.41
2,L,0.40
3,D,0.29


### 10. Teams whose leading scorer has more goals than the league-wide average leading scorer (correlated subquery)

In [ ]:
pd.read_sql('''
SELECT t.team_name, MAX(ss.goals) AS top_scorer_goals
FROM skater_season_stats ss
JOIN teams t ON t.team_id = ss.team_id
GROUP BY t.team_name
HAVING MAX(ss.goals) > (
    SELECT AVG(max_goals) FROM (
        SELECT MAX(goals) AS max_goals
        FROM skater_season_stats
        GROUP BY team_id
    )
)
ORDER BY top_scorer_goals DESC;
''', conn)


,team_name,top_scorer_goals
0,Colorado Avalanche,53
1,Montréal Canadiens,51
2,Edmonton Oilers,48
3,San Jose Sharks,45
4,Minnesota Wild,45
5,Dallas Stars,45
6,Tampa Bay Lightning,44
7,Nashville Predators,42
8,Detroit Red Wings,41
9,Anaheim Ducks,41


### 11. Current win/loss streaks league-wide (simple filter + ORDER BY)

In [ ]:
pd.read_sql('''
SELECT t.team_name, s.streak_type, s.streak_count
FROM standings s JOIN teams t ON t.team_id = s.team_id
WHERE s.streak_type = 'W'
ORDER BY s.streak_count DESC;
''', conn)


,team_name,streak_type,streak_count
0,Dallas Stars,W,5
1,Washington Capitals,W,4
2,St. Louis Blues,W,4
3,Colorado Avalanche,W,3
4,Philadelphia Flyers,W,3
5,Vegas Golden Knights,W,3
6,Florida Panthers,W,3
7,Boston Bruins,W,2
8,Carolina Hurricanes,W,1
9,Minnesota Wild,W,1


### 12. Highest-scoring completed games (JOIN across games + teams)

In [ ]:
pd.read_sql('''
SELECT g.game_date, ht.team_name AS home, g.home_score, at.team_name AS away, g.away_score,
       (g.home_score + g.away_score) AS total_goals
FROM games g
JOIN teams ht ON ht.team_id = g.home_team_id
JOIN teams at ON at.team_id = g.away_team_id
WHERE g.game_state = 'OFF'
ORDER BY total_goals DESC
LIMIT 10;
''', conn)


,game_date,home,home_score,away,away_score,total_goals
0,2026-03-08,Buffalo Sabres,8,Tampa Bay Lightning,7,15
1,2026-05-03,Colorado Avalanche,9,Minnesota Wild,6,15
2,2026-04-01,Colorado Avalanche,6,Vancouver Canucks,8,14
3,2025-10-11,San Jose Sharks,6,Anaheim Ducks,7,13
4,2025-11-17,Florida Panthers,8,Vancouver Canucks,5,13
5,2025-12-04,Edmonton Oilers,9,Seattle Kraken,4,13
6,2025-12-06,Florida Panthers,7,Columbus Blue Jackets,6,13
7,2026-01-24,Columbus Blue Jackets,8,Tampa Bay Lightning,5,13
8,2026-04-04,Pittsburgh Penguins,9,Florida Panthers,4,13
9,2026-04-04,Los Angeles Kings,7,Toronto Maple Leafs,6,13


In [ ]:
# ============================================================
# Q13: CLASSIFY TEAMS BY POINT TOTAL
# ============================================================

query = """
SELECT
    t.team_name,
    s.points,
    CASE
        WHEN s.points >= 110 THEN '110+ Points'
        WHEN s.points >= 100 THEN '100-109 Points'
        WHEN s.points >= 90 THEN '90-99 Points'
        ELSE 'Below 90 Points'
    END AS points_category
FROM standings s
JOIN teams t
    ON t.team_id = s.team_id
ORDER BY s.points DESC;
"""

result = pd.read_sql(query, conn)

result

,team_name,points,points_category
0,Colorado Avalanche,121,110+ Points
1,Carolina Hurricanes,113,110+ Points
2,Dallas Stars,112,110+ Points
3,Buffalo Sabres,109,100-109 Points
4,Tampa Bay Lightning,106,100-109 Points
5,Montréal Canadiens,106,100-109 Points
6,Minnesota Wild,104,100-109 Points
7,Boston Bruins,100,100-109 Points
8,Ottawa Senators,99,90-99 Points
9,Pittsburgh Penguins,98,90-99 Points


In [ ]:
# ============================================================
# Q14: RANK TEAMS BY GOALS USING A CTE
# ============================================================

query = """
WITH team_goals AS (
    SELECT
        t.team_name,
        s.goals_for
    FROM standings s
    JOIN teams t
        ON t.team_id = s.team_id
)
SELECT
    team_name,
    goals_for
FROM team_goals
ORDER BY goals_for DESC;
"""

result = pd.read_sql(query, conn)

result

,team_name,goals_for
0,Colorado Avalanche,302
1,Carolina Hurricanes,296
2,Pittsburgh Penguins,293
3,Tampa Bay Lightning,290
4,Buffalo Sabres,288
5,Montréal Canadiens,283
6,Edmonton Oilers,282
7,Dallas Stars,279
8,Ottawa Senators,278
9,Anaheim Ducks,273


In [ ]:
# ============================================================
# Q15: RANK PLAYERS BY SEASON POINTS
# ============================================================

query = """
SELECT
    p.first_name,
    p.last_name,
    t.team_name,
    ss.points,
    RANK() OVER (
        ORDER BY ss.points DESC
    ) AS points_rank
FROM skater_season_stats ss
JOIN players p
    ON p.player_id = ss.player_id
JOIN teams t
    ON t.team_id = ss.team_id
ORDER BY points_rank
LIMIT 20;
"""

result = pd.read_sql(query, conn)

result

,first_name,last_name,team_name,points,points_rank
0,Connor,McDavid,Edmonton Oilers,138,1
1,Nikita,Kucherov,Tampa Bay Lightning,130,2
2,Nathan,MacKinnon,Colorado Avalanche,127,3
3,Macklin,Celebrini,San Jose Sharks,115,4
4,Mark,Scheifele,Winnipeg Jets,103,5
5,Nick,Suzuki,Montréal Canadiens,101,6
6,Martin,Necas,Colorado Avalanche,100,7
7,David,Pastrnak,Boston Bruins,100,7
8,Leon,Draisaitl,Edmonton Oilers,97,9
9,Jason,Robertson,Dallas Stars,96,10


In [ ]:
# ============================================================
# Q16: RANK PLAYERS WITHIN THEIR POSITION
# ============================================================

query = """
SELECT
    p.first_name,
    p.last_name,
    p.position,
    t.team_name,
    ss.points,
    RANK() OVER (
        PARTITION BY p.position
        ORDER BY ss.points DESC
    ) AS position_rank
FROM skater_season_stats ss
JOIN players p
    ON p.player_id = ss.player_id
JOIN teams t
    ON t.team_id = ss.team_id
ORDER BY p.position, position_rank
LIMIT 30;
"""

result = pd.read_sql(query, conn)

result

,first_name,last_name,position,team_name,points,position_rank
0,Connor,McDavid,C,Edmonton Oilers,138,1
1,Nathan,MacKinnon,C,Colorado Avalanche,127,2
2,Macklin,Celebrini,C,San Jose Sharks,115,3
3,Mark,Scheifele,C,Winnipeg Jets,103,4
4,Nick,Suzuki,C,Montréal Canadiens,101,5
5,Martin,Necas,C,Colorado Avalanche,100,6
6,Leon,Draisaitl,C,Edmonton Oilers,97,7
7,Jack,Eichel,C,Vegas Golden Knights,90,8
8,Jake,Guentzel,C,Tampa Bay Lightning,88,9
9,Wyatt,Johnston,C,Dallas Stars,86,10


In [ ]:
# ============================================================
# Q17: TOP POINT SCORER FROM EVERY TEAM
# ============================================================

query = """
WITH ranked_players AS (
    SELECT
        ss.team_id,
        p.first_name,
        p.last_name,
        t.team_name,
        ss.points,
        ROW_NUMBER() OVER (
            PARTITION BY ss.team_id
            ORDER BY ss.points DESC, p.last_name
        ) AS player_rank
    FROM skater_season_stats ss
    JOIN players p
        ON p.player_id = ss.player_id
    JOIN teams t
        ON t.team_id = ss.team_id
)
SELECT
    team_name,
    first_name,
    last_name,
    points
FROM ranked_players
WHERE player_rank = 1
ORDER BY points DESC;
"""

result = pd.read_sql(query, conn)

result

,team_name,first_name,last_name,points
0,Edmonton Oilers,Connor,McDavid,138
1,Tampa Bay Lightning,Nikita,Kucherov,130
2,Colorado Avalanche,Nathan,MacKinnon,127
3,San Jose Sharks,Macklin,Celebrini,115
4,Winnipeg Jets,Mark,Scheifele,103
5,Montréal Canadiens,Nick,Suzuki,101
6,Boston Bruins,David,Pastrnak,100
7,Dallas Stars,Jason,Robertson,96
8,Vegas Golden Knights,Jack,Eichel,90
9,Minnesota Wild,Kirill,Kaprizov,89


In [ ]:
# ============================================================
# Q18: SCORING BY GAME TYPE
# ============================================================

query = """
SELECT
    CASE
        WHEN game_type = 1 THEN 'Preseason'
        WHEN game_type = 2 THEN 'Regular Season'
        WHEN game_type = 3 THEN 'Postseason'
        ELSE 'Other'
    END AS game_type_name,
    COUNT(*) AS games_played,
    SUM(home_score + away_score) AS total_goals,
    ROUND(
        AVG(home_score + away_score),
        2
    ) AS avg_goals_per_game
FROM games
WHERE game_state = 'OFF'
GROUP BY game_type
ORDER BY game_type;
"""

result = pd.read_sql(query, conn)

result

,game_type_name,games_played,total_goals,avg_goals_per_game
0,Regular Season,1312,8205,6.25
1,Postseason,82,486,5.93


In [ ]:
# ============================================================
# Q19: HIGHEST SINGLE-GAME PLAYER POINT TOTALS
# ============================================================

query = """
SELECT
    p.first_name,
    p.last_name,
    COUNT(*) AS games_recorded,
    MAX(gs.points) AS highest_game_points,
    SUM(gs.points) AS total_game_points
FROM game_stats gs
JOIN players p
    ON p.player_id = gs.player_id
GROUP BY gs.player_id
ORDER BY highest_game_points DESC, total_game_points DESC
LIMIT 20;
"""

result = pd.read_sql(query, conn)

result

,first_name,last_name,games_recorded,highest_game_points,total_game_points
0,David,Pastrnak,85,6,109
1,Evan,Bouchard,91,6,105
2,Connor,McDavid,91,5,148
3,Nathan,MacKinnon,95,5,143
4,Nikita,Kucherov,86,5,140
5,Macklin,Celebrini,83,5,116
6,Leon,Draisaitl,74,5,112
7,Mitch,Marner,106,5,110
8,Cole,Caufield,103,5,105
9,Tage,Thompson,98,5,103


In [ ]:
# ============================================================
# Q20: TOP GOAL SCORER AND SHARE OF TEAM GOALS
# ============================================================

query = """
WITH ranked_scorers AS (
    SELECT
        ss.team_id,
        ss.player_id,
        ss.goals,
        ROW_NUMBER() OVER (
            PARTITION BY ss.team_id
            ORDER BY ss.goals DESC, p.last_name
        ) AS scorer_rank
    FROM skater_season_stats ss
    JOIN players p
        ON p.player_id = ss.player_id
),
top_scorers AS (
    SELECT
        team_id,
        player_id,
        goals
    FROM ranked_scorers
    WHERE scorer_rank = 1
)
SELECT
    t.team_name,
    p.first_name,
    p.last_name,
    ts.goals AS top_scorer_goals,
    s.goals_for AS team_goals,
    ROUND(
        ts.goals * 100.0 / NULLIF(s.goals_for, 0),
        2
    ) AS percentage_of_team_goals
FROM top_scorers ts
JOIN teams t
    ON t.team_id = ts.team_id
JOIN players p
    ON p.player_id = ts.player_id
JOIN standings s
    ON s.team_id = ts.team_id
ORDER BY percentage_of_team_goals DESC;
"""

result = pd.read_sql(query, conn)

result

,team_name,first_name,last_name,top_scorer_goals,team_goals,percentage_of_team_goals
0,Montréal Canadiens,Cole,Caufield,51,283,18.02
1,San Jose Sharks,Macklin,Celebrini,45,251,17.93
2,Colorado Avalanche,Nathan,MacKinnon,53,302,17.55
3,Edmonton Oilers,Connor,McDavid,48,282,17.02
4,Detroit Red Wings,Alex,DeBrincat,41,241,17.01
5,Nashville Predators,Steven,Stamkos,42,247,17.00
6,Winnipeg Jets,Kyle,Connor,39,231,16.88
7,Minnesota Wild,Kirill,Kaprizov,45,272,16.54
8,Dallas Stars,Wyatt,Johnston,45,279,16.13
9,Los Angeles Kings,Adrian,Kempe,36,225,16.00


In [ ]:
# ============================================================
# Q21: AVERAGE PLAYER POINTS BY TEAM
# ============================================================

query = """
SELECT
    t.team_name,
    COUNT(ss.player_id) AS players_with_stats,
    ROUND(AVG(ss.points), 2) AS avg_player_points
FROM skater_season_stats ss
JOIN teams t
    ON t.team_id = ss.team_id
GROUP BY t.team_id, t.team_name
ORDER BY avg_player_points DESC;
"""

result = pd.read_sql(query, conn)

result

,team_name,players_with_stats,avg_player_points
0,Dallas Stars,26,28.73
1,Colorado Avalanche,29,28.17
2,Montréal Canadiens,26,27.81
3,Utah Mammoth,26,27.35
4,Anaheim Ducks,26,26.88
5,Washington Capitals,27,26.67
6,Carolina Hurricanes,29,26.38
7,Buffalo Sabres,29,25.90
8,Edmonton Oilers,30,25.57
9,Vegas Golden Knights,28,25.50


In [ ]:
# ============================================================
# Q22: TOP 3 POINT SCORERS FROM EVERY TEAM
# ============================================================

query = """
WITH ranked_players AS (
    SELECT
        ss.team_id,
        t.team_name,
        p.first_name,
        p.last_name,
        ss.points,
        RANK() OVER (
            PARTITION BY ss.team_id
            ORDER BY ss.points DESC
        ) AS team_rank
    FROM skater_season_stats ss
    JOIN players p
        ON p.player_id = ss.player_id
    JOIN teams t
        ON t.team_id = ss.team_id
)
SELECT
    team_name,
    first_name,
    last_name,
    points,
    team_rank
FROM ranked_players
WHERE team_rank <= 3
ORDER BY team_name, team_rank, points DESC;
"""

result = pd.read_sql(query, conn)

result

,team_name,first_name,last_name,points,team_rank
0,Anaheim Ducks,Cutter,Gauthier,69,1
1,Anaheim Ducks,Leo,Carlsson,67,2
2,Anaheim Ducks,Beckett,Sennecke,60,3
3,Boston Bruins,David,Pastrnak,100,1
4,Boston Bruins,Morgan,Geekie,68,2
...,...,...,...,...,...
95,Washington Capitals,Jakob,Chychrun,60,3
96,Washington Capitals,John,Carlson,60,3
97,Winnipeg Jets,Mark,Scheifele,103,1
98,Winnipeg Jets,Kyle,Connor,92,2


In [ ]:
# ============================================================
# Q23: PLAYERS WITH POINTS IN BOTH REGULAR SEASON
#      AND POSTSEASON
# ============================================================

query = """
SELECT
    p.first_name,
    p.last_name,
    SUM(
        CASE
            WHEN g.game_type = 2 THEN gs.points
            ELSE 0
        END
    ) AS regular_season_points,
    SUM(
        CASE
            WHEN g.game_type = 3 THEN gs.points
            ELSE 0
        END
    ) AS postseason_points
FROM game_stats gs
JOIN games g
    ON g.game_id = gs.game_id
JOIN players p
    ON p.player_id = gs.player_id
WHERE g.game_state = 'OFF'
  AND g.game_type IN (2, 3)
GROUP BY gs.player_id
HAVING
    SUM(
        CASE
            WHEN g.game_type = 2 THEN gs.points
            ELSE 0
        END
    ) > 0
    AND
    SUM(
        CASE
            WHEN g.game_type = 3 THEN gs.points
            ELSE 0
        END
    ) > 0
ORDER BY regular_season_points DESC;
"""

result = pd.read_sql(query, conn)

result

,first_name,last_name,regular_season_points,postseason_points
0,Connor,McDavid,138,6
1,Nikita,Kucherov,130,6
2,Nathan,MacKinnon,127,15
3,Nick,Suzuki,101,16
4,Martin,Necas,100,13
...,...,...,...,...
250,Arber,Xhekaj,4,2
251,Liam,O'Brien,4,1
252,Garnet,Hathaway,3,2
253,Carter,Yakemchuk,2,2


In [ ]:
# ============================================================
# Q24: TOP 3 SCORERS' SHARE OF TEAM GOALS
# ============================================================

query = """
WITH ranked_scorers AS (
    SELECT
        ss.team_id,
        ss.player_id,
        ss.goals,
        RANK() OVER (
            PARTITION BY ss.team_id
            ORDER BY ss.goals DESC
        ) AS goal_rank
    FROM skater_season_stats ss
),
top_three AS (
    SELECT
        team_id,
        SUM(goals) AS top_three_goals
    FROM ranked_scorers
    WHERE goal_rank <= 3
    GROUP BY team_id
)
SELECT
    t.team_name,
    tt.top_three_goals,
    s.goals_for AS team_goals,
    ROUND(
        tt.top_three_goals * 100.0 /
        NULLIF(s.goals_for, 0),
        2
    ) AS top_three_goal_share_pct
FROM top_three tt
JOIN teams t
    ON t.team_id = tt.team_id
JOIN standings s
    ON s.team_id = tt.team_id
ORDER BY top_three_goal_share_pct DESC;
"""

result = pd.read_sql(query, conn)

result

,team_name,top_three_goals,team_goals,top_three_goal_share_pct
0,Minnesota Wild,133,272,48.90
1,Winnipeg Jets,105,231,45.45
2,Nashville Predators,107,247,43.32
3,Detroit Red Wings,100,241,41.49
4,Colorado Avalanche,124,302,41.06
5,Tampa Bay Lightning,118,290,40.69
6,Edmonton Oilers,114,282,40.43
7,Dallas Stars,112,279,40.14
8,Montréal Canadiens,110,283,38.87
9,Chicago Blackhawks,80,213,37.56


In [ ]:
# ============================================================
# Q25: HOME VS AWAY WIN PERCENTAGES
# ============================================================

query = """
SELECT
    t.team_name,
    s.games_played,
    s.home_wins,
    s.away_wins,
    ROUND(
        s.home_wins * 100.0 /
        NULLIF(s.games_played, 0),
        2
    ) AS home_win_pct,
    ROUND(
        s.away_wins * 100.0 /
        NULLIF(s.games_played, 0),
        2
    ) AS away_win_pct
FROM standings s
JOIN teams t
    ON t.team_id = s.team_id
ORDER BY home_win_pct DESC;
"""

result = pd.read_sql(query, conn)

result

,team_name,games_played,home_wins,away_wins,home_win_pct,away_win_pct
0,Carolina Hurricanes,82,29,24,35.37,29.27
1,Boston Bruins,82,29,16,35.37,19.51
2,Colorado Avalanche,82,26,29,31.71,35.37
3,Dallas Stars,82,26,24,31.71,29.27
4,Buffalo Sabres,82,26,24,31.71,29.27
5,Tampa Bay Lightning,82,26,24,31.71,29.27
6,Washington Capitals,82,25,18,30.49,21.95
7,Montréal Canadiens,82,24,24,29.27,29.27
8,Anaheim Ducks,82,24,19,29.27,23.17
9,Minnesota Wild,82,23,23,28.05,28.05


In [ ]:
# ================================================================
# UPDATE app/queries.sql WITH VALIDATED FINAL SQL QUERIES
# ================================================================

from pathlib import Path
from datetime import datetime

PROJECT_DIR = Path("/content/drive/MyDrive/NHL_Analytics_Hub")
QUERIES_PATH = PROJECT_DIR / "app" / "queries.sql"

# ------------------------------------------------
# 1. Verify project path
# ------------------------------------------------

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

print(f"Project directory: {PROJECT_DIR}")
print(f"queries.sql path:  {QUERIES_PATH}")

# ------------------------------------------------
# 2. Create backup of existing queries.sql
# ------------------------------------------------

if QUERIES_PATH.exists():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = QUERIES_PATH.with_name(
        f"queries_backup_{timestamp}.sql"
    )

    backup_path.write_text(
        QUERIES_PATH.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

    print(f"\nBackup created:")
    print(backup_path)
else:
    print("\nNo existing queries.sql found. A new file will be created.")

# ------------------------------------------------
# 3. Final validated SQL content
# ------------------------------------------------

queries_sql = r"""
-- NHL Analytics Hub — SQL Analysis Queries
-- Validated against the current NHL Analytics Hub SQLite database.
--
-- Purpose:
--   1. Store reusable SQL queries for the Streamlit dashboard.
--   2. Keep validated analytical SQL separate from app.py.
--   3. Serve as a clean reference for the project's SQL analytics layer.
--
-- Database:
--   SQLite
--
-- Current season:
--   20252026
--
-- Tables used:
--   teams
--   standings
--   players
--   games
--   game_stats
--   skater_season_stats
--   goalie_season_stats


-- ================================================================
-- TEAM ANALYTICS
-- ================================================================


-- Q1: Top 10 teams by goals
SELECT
    t.team_name,
    s.goals_for
FROM standings s
JOIN teams t
    ON t.team_id = s.team_id
ORDER BY s.goals_for DESC
LIMIT 10;


-- Q4: Teams above league-average points
SELECT
    t.team_name,
    s.points
FROM standings s
JOIN teams t
    ON t.team_id = s.team_id
WHERE s.points > (
    SELECT AVG(points)
    FROM standings
)
ORDER BY s.points DESC;


-- Q5: Divisions averaging above 90 points
SELECT
    t.division_name,
    ROUND(AVG(s.points), 2) AS avg_points
FROM standings s
JOIN teams t
    ON t.team_id = s.team_id
GROUP BY t.division_name
HAVING AVG(s.points) > 90
ORDER BY avg_points DESC;


-- Q8: Home vs away win difference
SELECT
    t.team_name,
    s.home_wins,
    s.away_wins,
    (s.home_wins - s.away_wins) AS home_minus_away
FROM standings s
JOIN teams t
    ON t.team_id = s.team_id
ORDER BY home_minus_away DESC;


-- Q10: Teams whose top scorer exceeds the average
--      top-scorer goal total across teams
SELECT
    t.team_name,
    MAX(ss.goals) AS top_scorer_goals
FROM skater_season_stats ss
JOIN teams t
    ON t.team_id = ss.team_id
GROUP BY t.team_name
HAVING MAX(ss.goals) > (
    SELECT AVG(max_goals)
    FROM (
        SELECT MAX(goals) AS max_goals
        FROM skater_season_stats
        GROUP BY team_id
    )
)
ORDER BY top_scorer_goals DESC;


-- Q11: Teams currently on a winning streak
SELECT
    t.team_name,
    s.streak_type,
    s.streak_count
FROM standings s
JOIN teams t
    ON t.team_id = s.team_id
WHERE s.streak_type = 'W'
ORDER BY s.streak_count DESC;


-- ================================================================
-- PLAYER ANALYTICS
-- ================================================================


-- Q2: Top 5 point scorers
SELECT
    p.first_name,
    p.last_name,
    t.team_name,
    ss.points
FROM skater_season_stats ss
JOIN players p
    ON p.player_id = ss.player_id
JOIN teams t
    ON t.team_id = ss.team_id
ORDER BY ss.points DESC
LIMIT 5;


-- Q3: Players with more than 20 goals and more than 30 assists
SELECT
    p.first_name,
    p.last_name,
    ss.goals,
    ss.assists
FROM skater_season_stats ss
JOIN players p
    ON p.player_id = ss.player_id
WHERE ss.goals > 20
  AND ss.assists > 30
ORDER BY ss.goals DESC;


-- Q6: Top goalies by save percentage,
--     minimum 10 games played
SELECT
    p.first_name,
    p.last_name,
    t.team_name,
    gs.save_pct,
    gs.games_played
FROM goalie_season_stats gs
JOIN players p
    ON p.player_id = gs.player_id
JOIN teams t
    ON t.team_id = gs.team_id
WHERE gs.games_played >= 10
ORDER BY gs.save_pct DESC
LIMIT 10;


-- Q7: Top 10 players by penalty minutes
SELECT
    p.first_name,
    p.last_name,
    t.team_name,
    ss.penalty_min
FROM skater_season_stats ss
JOIN players p
    ON p.player_id = ss.player_id
JOIN teams t
    ON t.team_id = ss.team_id
ORDER BY ss.penalty_min DESC
LIMIT 10;


-- Q9: Average points per game by position
SELECT
    p.position,
    ROUND(
        AVG(
            ss.points * 1.0 /
            NULLIF(ss.games_played, 0)
        ),
        2
    ) AS avg_points_per_game
FROM skater_season_stats ss
JOIN players p
    ON p.player_id = ss.player_id
WHERE ss.games_played > 0
GROUP BY p.position
ORDER BY avg_points_per_game DESC;


-- Q17: Top point scorer from every team
WITH ranked_players AS (
    SELECT
        ss.team_id,
        p.first_name,
        p.last_name,
        t.team_name,
        ss.points,
        ROW_NUMBER() OVER (
            PARTITION BY ss.team_id
            ORDER BY ss.points DESC, p.last_name
        ) AS player_rank
    FROM skater_season_stats ss
    JOIN players p
        ON p.player_id = ss.player_id
    JOIN teams t
        ON t.team_id = ss.team_id
)
SELECT
    team_name,
    first_name,
    last_name,
    points
FROM ranked_players
WHERE player_rank = 1
ORDER BY points DESC;


-- Q22: Top 3 point scorers per team
--
-- RANK() intentionally allows ties to be included.
WITH ranked_players AS (
    SELECT
        ss.team_id,
        t.team_name,
        p.first_name,
        p.last_name,
        ss.points,
        RANK() OVER (
            PARTITION BY ss.team_id
            ORDER BY ss.points DESC
        ) AS team_rank
    FROM skater_season_stats ss
    JOIN players p
        ON p.player_id = ss.player_id
    JOIN teams t
        ON t.team_id = ss.team_id
)
SELECT
    team_name,
    first_name,
    last_name,
    points,
    team_rank
FROM ranked_players
WHERE team_rank <= 3
ORDER BY team_name, team_rank, points DESC;


-- ================================================================
-- GAME ANALYTICS
-- ================================================================


-- Q12: Highest-scoring completed games
SELECT
    g.game_date,
    ht.team_name AS home,
    g.home_score,
    at.team_name AS away,
    g.away_score,
    (g.home_score + g.away_score) AS total_goals
FROM games g
JOIN teams ht
    ON ht.team_id = g.home_team_id
JOIN teams at
    ON at.team_id = g.away_team_id
WHERE g.game_state = 'OFF'
ORDER BY total_goals DESC
LIMIT 10;


-- Q18: Scoring by game type
SELECT
    CASE
        WHEN game_type = 1 THEN 'Preseason'
        WHEN game_type = 2 THEN 'Regular Season'
        WHEN game_type = 3 THEN 'Postseason'
        ELSE 'Other'
    END AS game_type_name,
    COUNT(*) AS games_played,
    SUM(home_score + away_score) AS total_goals,
    ROUND(
        AVG(home_score + away_score),
        2
    ) AS avg_goals_per_game
FROM games
WHERE game_state = 'OFF'
GROUP BY game_type
ORDER BY game_type;


-- Q23: Players with points in both regular season
--      and postseason
SELECT
    p.first_name,
    p.last_name,
    SUM(
        CASE
            WHEN g.game_type = 2 THEN gs.points
            ELSE 0
        END
    ) AS regular_season_points,
    SUM(
        CASE
            WHEN g.game_type = 3 THEN gs.points
            ELSE 0
        END
    ) AS postseason_points
FROM game_stats gs
JOIN games g
    ON g.game_id = gs.game_id
JOIN players p
    ON p.player_id = gs.player_id
WHERE g.game_state = 'OFF'
  AND g.game_type IN (2, 3)
GROUP BY gs.player_id
HAVING
    SUM(
        CASE
            WHEN g.game_type = 2 THEN gs.points
            ELSE 0
        END
    ) > 0
    AND
    SUM(
        CASE
            WHEN g.game_type = 3 THEN gs.points
            ELSE 0
        END
    ) > 0
ORDER BY regular_season_points DESC;


-- ================================================================
-- ADVANCED ANALYTICS
-- ================================================================


-- Q13: Categorize teams by points
SELECT
    t.team_name,
    s.points,
    CASE
        WHEN s.points >= 110 THEN '110+ Points'
        WHEN s.points >= 100 THEN '100-109 Points'
        WHEN s.points >= 90 THEN '90-99 Points'
        ELSE 'Below 90 Points'
    END AS points_category
FROM standings s
JOIN teams t
    ON t.team_id = s.team_id
ORDER BY s.points DESC;


-- Q15: League-wide player ranking by points
SELECT
    p.first_name,
    p.last_name,
    t.team_name,
    ss.points,
    RANK() OVER (
        ORDER BY ss.points DESC
    ) AS points_rank
FROM skater_season_stats ss
JOIN players p
    ON p.player_id = ss.player_id
JOIN teams t
    ON t.team_id = ss.team_id
ORDER BY points_rank
LIMIT 20;


-- Q16: Player ranking by position
SELECT
    p.first_name,
    p.last_name,
    p.position,
    t.team_name,
    ss.points,
    RANK() OVER (
        PARTITION BY p.position
        ORDER BY ss.points DESC
    ) AS position_rank
FROM skater_season_stats ss
JOIN players p
    ON p.player_id = ss.player_id
JOIN teams t
    ON t.team_id = ss.team_id
ORDER BY p.position, position_rank
LIMIT 30;


-- Q20: Top scorer's percentage of team goals
WITH ranked_scorers AS (
    SELECT
        ss.team_id,
        ss.player_id,
        ss.goals,
        ROW_NUMBER() OVER (
            PARTITION BY ss.team_id
            ORDER BY ss.goals DESC, p.last_name
        ) AS scorer_rank
    FROM skater_season_stats ss
    JOIN players p
        ON p.player_id = ss.player_id
),
top_scorers AS (
    SELECT
        team_id,
        player_id,
        goals
    FROM ranked_scorers
    WHERE scorer_rank = 1
)
SELECT
    t.team_name,
    p.first_name,
    p.last_name,
    ts.goals AS top_scorer_goals,
    s.goals_for AS team_goals,
    ROUND(
        ts.goals * 100.0 /
        NULLIF(s.goals_for, 0),
        2
    ) AS percentage_of_team_goals
FROM top_scorers ts
JOIN teams t
    ON t.team_id = ts.team_id
JOIN players p
    ON p.player_id = ts.player_id
JOIN standings s
    ON s.team_id = ts.team_id
ORDER BY percentage_of_team_goals DESC;


-- Q24: Top 3 scorers' share of team goals
--
-- RANK() intentionally includes ties at the third rank.
WITH ranked_scorers AS (
    SELECT
        ss.team_id,
        ss.player_id,
        ss.goals,
        RANK() OVER (
            PARTITION BY ss.team_id
            ORDER BY ss.goals DESC
        ) AS goal_rank
    FROM skater_season_stats ss
),
top_three AS (
    SELECT
        team_id,
        SUM(goals) AS top_three_goals
    FROM ranked_scorers
    WHERE goal_rank <= 3
    GROUP BY team_id
)
SELECT
    t.team_name,
    tt.top_three_goals,
    s.goals_for AS team_goals,
    ROUND(
        tt.top_three_goals * 100.0 /
        NULLIF(s.goals_for, 0),
        2
    ) AS top_three_goal_share_pct
FROM top_three tt
JOIN teams t
    ON t.team_id = tt.team_id
JOIN standings s
    ON s.team_id = tt.team_id
ORDER BY top_three_goal_share_pct DESC;
"""

# ------------------------------------------------
# 4. Write the new queries.sql
# ------------------------------------------------

QUERIES_PATH.parent.mkdir(parents=True, exist_ok=True)

QUERIES_PATH.write_text(
    queries_sql.strip() + "\n",
    encoding="utf-8"
)

# ------------------------------------------------
# 5. Verify
# ------------------------------------------------

written_text = QUERIES_PATH.read_text(encoding="utf-8")

query_numbers = [
    line.strip()
    for line in written_text.splitlines()
    if line.strip().startswith("-- Q")
]

print("\n" + "=" * 70)
print("queries.sql UPDATED SUCCESSFULLY")
print("=" * 70)

print(f"\nFile:")
print(QUERIES_PATH)

print(f"\nFile size:")
print(f"{QUERIES_PATH.stat().st_size:,} bytes")

print(f"\nQuery sections written:")
print(len(query_numbers))

print("\nQueries:")
for q in query_numbers:
    print("  " + q)

print("\nBackup:")
if 'backup_path' in locals():
    print(backup_path)
else:
    print("No previous file existed.")

print("\nStatus: SUCCESS")

Project directory: /content/drive/MyDrive/NHL_Analytics_Hub
queries.sql path:  /content/drive/MyDrive/NHL_Analytics_Hub/app/queries.sql

No existing queries.sql found. A new file will be created.

queries.sql UPDATED SUCCESSFULLY

File:
/content/drive/MyDrive/NHL_Analytics_Hub/app/queries.sql

File size:
10,089 bytes

Query sections written:
21

Queries:
  -- Q1: Top 10 teams by goals
  -- Q4: Teams above league-average points
  -- Q5: Divisions averaging above 90 points
  -- Q8: Home vs away win difference
  -- Q10: Teams whose top scorer exceeds the average
  -- Q11: Teams currently on a winning streak
  -- Q2: Top 5 point scorers
  -- Q3: Players with more than 20 goals and more than 30 assists
  -- Q6: Top goalies by save percentage,
  -- Q7: Top 10 players by penalty minutes
  -- Q9: Average points per game by position
  -- Q17: Top point scorer from every team
  -- Q22: Top 3 point scorers per team
  -- Q12: Highest-scoring completed games
  -- Q18: Scoring by game type
  -- Q23:

In [ ]:
# ================================================================
# VALIDATE app/queries.sql AGAINST THE CURRENT DATABASE
# ================================================================

from pathlib import Path
import sqlite3

PROJECT_DIR = Path("/content/drive/MyDrive/NHL_Analytics_Hub")
DB_PATH = PROJECT_DIR / "data" / "nhl_hockey.db"
QUERIES_PATH = PROJECT_DIR / "app" / "queries.sql"

print("=" * 70)
print("NHL ANALYTICS HUB - queries.sql VALIDATION")
print("=" * 70)

# ------------------------------------------------
# 1. Verify files exist
# ------------------------------------------------

print("\n[1] FILE CHECK")

print(f"Database : {DB_PATH}")
print(f"queries  : {QUERIES_PATH}")

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

if not QUERIES_PATH.exists():
    raise FileNotFoundError(f"queries.sql not found: {QUERIES_PATH}")

print("Database: FOUND")
print("queries.sql: FOUND")


# ------------------------------------------------
# 2. Read queries.sql
# ------------------------------------------------

sql_text = QUERIES_PATH.read_text(encoding="utf-8")

print("\n[2] SQL FILE CHECK")

print(f"File size: {len(sql_text.encode('utf-8')):,} bytes")

# ------------------------------------------------
# 3. Split SQL file into individual queries
#
# Each query starts with:
# -- Q<number>:
# ------------------------------------------------

import re

parts = re.split(
    r'(?=-- Q\d+:)',
    sql_text
)

queries = []

for part in parts:
    part = part.strip()

    if not part.startswith("-- Q"):
        continue

    # Find query number/title
    match = re.match(r'(-- Q\d+:.*?)(?:\n|$)', part)

    if not match:
        continue

    title = match.group(1).strip()

    # Remove comments before SQL
    sql_lines = []

    for line in part.splitlines():
        stripped = line.strip()

        if stripped.startswith("--"):
            continue

        if stripped:
            sql_lines.append(line)

    query_sql = "\n".join(sql_lines).strip()

    if query_sql:
        queries.append({
            "title": title,
            "sql": query_sql
        })


print(f"Queries detected: {len(queries)}")

for i, q in enumerate(queries, start=1):
    print(f"  {i:02d}. {q['title']}")


# ------------------------------------------------
# 4. Execute every query
# ------------------------------------------------

print("\n[3] QUERY EXECUTION VALIDATION")

conn_test = sqlite3.connect(DB_PATH)

passed = 0
failed = 0

results = []

for q in queries:
    try:
        cursor = conn_test.execute(q["sql"])
        rows = cursor.fetchall()

        passed += 1

        results.append({
            "query": q["title"],
            "status": "PASS",
            "rows": len(rows)
        })

        print(f"PASS  {q['title']}  → {len(rows)} rows")

    except Exception as e:
        failed += 1

        results.append({
            "query": q["title"],
            "status": "FAIL",
            "rows": 0
        })

        print(f"FAIL  {q['title']}")
        print(f"      Error: {e}")


conn_test.close()


# ------------------------------------------------
# 5. Final summary
# ------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

print(f"Total queries : {len(queries)}")
print(f"Passed        : {passed}")
print(f"Failed        : {failed}")

if failed == 0:
    print("\nSTATUS: ALL QUERIES PASSED")
else:
    print("\nSTATUS: SOME QUERIES FAILED")
    print("Review the errors above before modifying app.py.")

NHL ANALYTICS HUB - queries.sql VALIDATION

[1] FILE CHECK
Database : /content/drive/MyDrive/NHL_Analytics_Hub/data/nhl_hockey.db
queries  : /content/drive/MyDrive/NHL_Analytics_Hub/app/queries.sql
Database: FOUND
queries.sql: FOUND

[2] SQL FILE CHECK
File size: 10,089 bytes
Queries detected: 21
  01. -- Q1: Top 10 teams by goals
  02. -- Q4: Teams above league-average points
  03. -- Q5: Divisions averaging above 90 points
  04. -- Q8: Home vs away win difference
  05. -- Q10: Teams whose top scorer exceeds the average
  06. -- Q11: Teams currently on a winning streak
  07. -- Q2: Top 5 point scorers
  08. -- Q3: Players with more than 20 goals and more than 30 assists
  09. -- Q6: Top goalies by save percentage,
  10. -- Q7: Top 10 players by penalty minutes
  11. -- Q9: Average points per game by position
  12. -- Q17: Top point scorer from every team
  13. -- Q22: Top 3 point scorers per team
  14. -- Q12: Highest-scoring completed games
  15. -- Q18: Scoring by game type
  16. --

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/NHL_Analytics_Hub")

print("=" * 70)
print("NHL ANALYTICS HUB - LOCATING STREAMLIT APP")
print("=" * 70)

print(f"\nProject directory: {PROJECT_DIR}")
print(f"Project exists   : {PROJECT_DIR.exists()}")

print("\n[1] Python files in project:")

python_files = sorted(PROJECT_DIR.rglob("*.py"))

if python_files:
    for path in python_files:
        print("  ", path.relative_to(PROJECT_DIR))
else:
    print("  No .py files found.")

print("\n[2] Files under app/:")

APP_DIR = PROJECT_DIR / "app"

if APP_DIR.exists():
    for path in sorted(APP_DIR.rglob("*")):
        if path.is_file():
            print("  ", path.relative_to(PROJECT_DIR))
else:
    print("  app/ folder does not exist.")

print("\n[3] Possible Streamlit files:")

streamlit_candidates = []

for path in python_files:
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    if "streamlit" in text.lower() or "st." in text:
        streamlit_candidates.append(path)

if streamlit_candidates:
    for path in streamlit_candidates:
        print("  ", path.relative_to(PROJECT_DIR))
else:
    print("  No Streamlit Python file detected.")

print("\n" + "=" * 70)

NHL ANALYTICS HUB - LOCATING STREAMLIT APP

Project directory: /content/drive/MyDrive/NHL_Analytics_Hub
Project exists   : True

[1] Python files in project:
   app.py
   nhl_api_utils.py

[2] Files under app/:
   app/queries.sql

[3] Possible Streamlit files:
   app.py



In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/NHL_Analytics_Hub")
APP_PATH = PROJECT_DIR / "app.py"

print("=" * 70)
print("NHL ANALYTICS HUB - CURRENT STREAMLIT APP")
print("=" * 70)

print(f"\nApp path: {APP_PATH}")
print(f"Exists  : {APP_PATH.exists()}")

if APP_PATH.exists():
    app_text = APP_PATH.read_text(encoding="utf-8")

    print(f"File size: {len(app_text):,} characters")
    print(f"Lines    : {len(app_text.splitlines())}")

    print("\n" + "=" * 70)
    print("CURRENT app.py")
    print("=" * 70)

    print(app_text)
else:
    print("\nERROR: app.py was not found.")

NHL ANALYTICS HUB - CURRENT STREAMLIT APP

App path: /content/drive/MyDrive/NHL_Analytics_Hub/app.py
Exists  : True
File size: 15,993 characters
Lines    : 395

CURRENT app.py
"""
NHL Analytics Hub — Streamlit Dashboard
Run locally with:  streamlit run app.py

Expects nhl_hockey.db (built by the notebooks/ pipeline) to sit in ./data/
"""

import sqlite3
import pandas as pd
import streamlit as st
from streamlit_option_menu import option_menu

DB_PATH = "data/nhl_hockey.db"

st.set_page_config(page_title="NHL Analytics Hub", page_icon="🏒", layout="wide")


# ---------------------------------------------------------------- db helpers

@st.cache_resource
def get_connection():
    return sqlite3.connect(DB_PATH, check_same_thread=False)


def run_query(sql, params=None):
    conn = get_connection()
    try:
        return pd.read_sql(sql, conn, params=params)
    except Exception as e:
        st.error(f"SQL error: {e}")
        return pd.DataFrame()


# ------------------------------------

In [ ]:
from pathlib import Path
from datetime import datetime

PROJECT_DIR = Path("/content/drive/MyDrive/NHL_Analytics_Hub")
APP_PATH = PROJECT_DIR / "app.py"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_PATH = PROJECT_DIR / f"app_backup_{timestamp}.py"

APP_PATH.replace(BACKUP_PATH)

print("=" * 70)
print("NHL ANALYTICS HUB - app.py BACKUP")
print("=" * 70)

print(f"\nOriginal : {APP_PATH}")
print(f"Backup   : {BACKUP_PATH}")

# Restore the original file immediately after making the backup.
BACKUP_PATH.replace(APP_PATH)

print("\nBackup created successfully.")
print(f"Current app.py exists: {APP_PATH.exists()}")
print(f"Backup exists        : {BACKUP_PATH.exists()}")

NHL ANALYTICS HUB - app.py BACKUP

Original : /content/drive/MyDrive/NHL_Analytics_Hub/app.py
Backup   : /content/drive/MyDrive/NHL_Analytics_Hub/app_backup_20260922_124147.py

Backup created successfully.
Current app.py exists: True
Backup exists        : False


In [ ]:
from pathlib import Path
from shutil import copy2
from datetime import datetime

PROJECT_DIR = Path("/content/drive/MyDrive/NHL_Analytics_Hub")
APP_PATH = PROJECT_DIR / "app.py"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_PATH = PROJECT_DIR / f"app_backup_{timestamp}.py"

print("=" * 70)
print("NHL ANALYTICS HUB - CREATE REAL app.py BACKUP")
print("=" * 70)

print(f"\nOriginal app.py : {APP_PATH}")

if not APP_PATH.exists():
    raise FileNotFoundError(f"app.py not found: {APP_PATH}")

# Copy without modifying or moving the original.
copy2(APP_PATH, BACKUP_PATH)

print(f"Backup created : {BACKUP_PATH}")

print("\n[VERIFICATION]")
print(f"Original exists : {APP_PATH.exists()}")
print(f"Backup exists   : {BACKUP_PATH.exists()}")

if APP_PATH.exists() and BACKUP_PATH.exists():
    original_size = APP_PATH.stat().st_size
    backup_size = BACKUP_PATH.stat().st_size

    print(f"Original size   : {original_size:,} bytes")
    print(f"Backup size     : {backup_size:,} bytes")

    if original_size == backup_size:
        print("\nSTATUS: BACKUP CREATED SUCCESSFULLY")
    else:
        print("\nWARNING: File sizes differ.")
else:
    print("\nSTATUS: BACKUP FAILED")

NHL ANALYTICS HUB - CREATE REAL app.py BACKUP

Original app.py : /content/drive/MyDrive/NHL_Analytics_Hub/app.py
Backup created : /content/drive/MyDrive/NHL_Analytics_Hub/app_backup_20260922_124237.py

[VERIFICATION]
Original exists : True
Backup exists   : True
Original size   : 16,011 bytes
Backup size     : 16,011 bytes

STATUS: BACKUP CREATED SUCCESSFULLY


In [ ]:
from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/NHL_Analytics_Hub")
APP_PATH = PROJECT_DIR / "app.py"
QUERIES_PATH = PROJECT_DIR / "app" / "queries.sql"
DB_PATH = PROJECT_DIR / "data" / "nhl_hockey.db"

print("=" * 70)
print("NHL ANALYTICS HUB - app.py PATH + SYNTAX VALIDATION")
print("=" * 70)

print("\n[1] PATH CHECK")
print(f"app.py       : {APP_PATH}")
print(f"queries.sql  : {QUERIES_PATH}")
print(f"database     : {DB_PATH}")

print(f"\napp.py exists      : {APP_PATH.exists()}")
print(f"queries.sql exists : {QUERIES_PATH.exists()}")
print(f"database exists    : {DB_PATH.exists()}")

print("\n[2] PYTHON SYNTAX CHECK")

try:
    app_text = APP_PATH.read_text(encoding="utf-8")
    ast.parse(app_text)
    print("app.py syntax: PASS")
except SyntaxError as e:
    print("app.py syntax: FAIL")
    print(e)
    raise

print("\n[3] REQUIRED PATH CONFIG CHECK")

required_strings = [
    'PROJECT_DIR = Path(__file__).resolve().parent',
    'DB_PATH = PROJECT_DIR / "data" / "nhl_hockey.db"',
    'QUERIES_PATH = PROJECT_DIR / "app" / "queries.sql"',
    "def load_prebuilt_queries():",
    "PREBUILT = load_prebuilt_queries()",
]

for item in required_strings:
    if item in app_text:
        print(f"PASS  {item}")
    else:
        print(f"FAIL  {item}")

print("\n[4] SQL FILE CHECK")

sql_text = QUERIES_PATH.read_text(encoding="utf-8")
query_count = sql_text.count("-- Q")

print(f"queries.sql size    : {len(sql_text):,} characters")
print(f"Query markers found : {query_count}")

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

NHL ANALYTICS HUB - app.py PATH + SYNTAX VALIDATION

[1] PATH CHECK
app.py       : /content/drive/MyDrive/NHL_Analytics_Hub/app.py
queries.sql  : /content/drive/MyDrive/NHL_Analytics_Hub/app/queries.sql
database     : /content/drive/MyDrive/NHL_Analytics_Hub/data/nhl_hockey.db

app.py exists      : True
queries.sql exists : True
database exists    : True

[2] PYTHON SYNTAX CHECK
app.py syntax: PASS

[3] REQUIRED PATH CONFIG CHECK
PASS  PROJECT_DIR = Path(__file__).resolve().parent
PASS  DB_PATH = PROJECT_DIR / "data" / "nhl_hockey.db"
PASS  QUERIES_PATH = PROJECT_DIR / "app" / "queries.sql"
PASS  def load_prebuilt_queries():
PASS  PREBUILT = load_prebuilt_queries()

[4] SQL FILE CHECK
queries.sql size    : 10,087 characters
Query markers found : 21

VALIDATION COMPLETE


In [ ]:
from pathlib import Path
import ast
import re

PROJECT_DIR = Path("/content/drive/MyDrive/NHL_Analytics_Hub")
APP_PATH = PROJECT_DIR / "app.py"
QUERIES_PATH = PROJECT_DIR / "app" / "queries.sql"

print("=" * 70)
print("NHL ANALYTICS HUB - STREAMLIT SQL INTEGRATION VALIDATION")
print("=" * 70)

# ---------------------------------------------------------------
# 1. Syntax
# ---------------------------------------------------------------

print("\n[1] PYTHON SYNTAX")

app_text = APP_PATH.read_text(encoding="utf-8")

try:
    ast.parse(app_text)
    print("PASS  app.py syntax is valid")
except SyntaxError as e:
    print("FAIL  app.py syntax error")
    print(e)
    raise

# ---------------------------------------------------------------
# 2. Path configuration
# ---------------------------------------------------------------

print("\n[2] PATH CONFIGURATION")

checks = {
    "Database path": 'DB_PATH = PROJECT_DIR / "data" / "nhl_hockey.db"',
    "Queries path": 'QUERIES_PATH = PROJECT_DIR / "app" / "queries.sql"',
    "Query loader": "def load_prebuilt_queries():",
    "Loaded PREBUILT": "PREBUILT = load_prebuilt_queries()",
}

for label, text in checks.items():
    if text in app_text:
        print(f"PASS  {label}")
    else:
        print(f"FAIL  {label}")

# ---------------------------------------------------------------
# 3. Check old hard-coded PREBUILT dictionary is gone
# ---------------------------------------------------------------

print("\n[3] OLD HARDCODED QUERY CHECK")

sql_query_section = app_text.split("# ================================================================== SQL QUERY", 1)

if len(sql_query_section) == 2:
    sql_section = sql_query_section[1]

    # The old implementation had a dictionary assignment inside
    # the SQL Query page.
    if "PREBUILT = {" in sql_section:
        print("FAIL  Old hard-coded PREBUILT dictionary still exists")
    else:
        print("PASS  Old hard-coded PREBUILT dictionary removed")
else:
    print("FAIL  SQL Query section not found")

# ---------------------------------------------------------------
# 4. Validate SQL file query count
# ---------------------------------------------------------------

print("\n[4] queries.sql")

sql_text = QUERIES_PATH.read_text(encoding="utf-8")

matches = re.findall(r"^-- Q\d+:.*$", sql_text, flags=re.MULTILINE)

print(f"Query markers found: {len(matches)}")

if len(matches) == 21:
    print("PASS  All 21 validated queries are present")
else:
    print(f"WARNING  Expected 21 queries, found {len(matches)}")

print("\nQuery labels:")

for i, label in enumerate(matches, start=1):
    print(f"  {i:02d}. {label}")

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

NHL ANALYTICS HUB - STREAMLIT SQL INTEGRATION VALIDATION

[1] PYTHON SYNTAX
PASS  app.py syntax is valid

[2] PATH CONFIGURATION
PASS  Database path
PASS  Queries path
PASS  Query loader
PASS  Loaded PREBUILT

[3] OLD HARDCODED QUERY CHECK
PASS  Old hard-coded PREBUILT dictionary removed

[4] queries.sql
Query markers found: 21
PASS  All 21 validated queries are present

Query labels:
  01. -- Q1: Top 10 teams by goals
  02. -- Q4: Teams above league-average points
  03. -- Q5: Divisions averaging above 90 points
  04. -- Q8: Home vs away win difference
  05. -- Q10: Teams whose top scorer exceeds the average
  06. -- Q11: Teams currently on a winning streak
  07. -- Q2: Top 5 point scorers
  08. -- Q3: Players with more than 20 goals and more than 30 assists
  09. -- Q6: Top goalies by save percentage,
  10. -- Q7: Top 10 players by penalty minutes
  11. -- Q9: Average points per game by position
  12. -- Q17: Top point scorer from every team
  13. -- Q22: Top 3 point scorers per team

In [ ]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!streamlit --version

/content/drive/MyDrive/NHL_Analytics_Hub
Streamlit, version 1.64.0


In [ ]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!pip install -r requirements.txt

/content/drive/MyDrive/NHL_Analytics_Hub
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 92.9 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!streamlit run app.py --server.headless true --server.port 8501 > streamlit.log 2>&1 &

/content/drive/MyDrive/NHL_Analytics_Hub


In [ ]:
!sleep 5
!cat streamlit.log



2026-09-22 13:00:00.167 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.11.101.39:8501



In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
!/content/cloudflared tunnel --url http://localhost:8501

2026-09-22T13:01:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-22T13:01:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-22T13:01:51Z INF +--------------------------------------------------------------------------------------------+
2026-09-22T13:01:51Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-22T13:01:51Z INF |  https://subsidiary-chevy-queen-officially.trycloudfla

In [ ]:
!ps aux | grep -E "streamlit|cloudflared" | grep -v grep

In [ ]:
!curl -I http://localhost:8501

curl: (7) Failed to connect to localhost port 8501 after 0 ms: Couldn't connect to server


In [ ]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub
!streamlit run app.py --server.headless true --server.port 8501 > streamlit.log 2>&1 &

/content/drive/MyDrive/NHL_Analytics_Hub


In [ ]:
!sleep 5
!cat streamlit.log

/bin/bash: line 1: streamlit: command not found


In [ ]:
!curl -I http://localhost:8501

curl: (7) Failed to connect to localhost port 8501 after 0 ms: Couldn't connect to server


In [ ]:
!ps aux | grep -E "streamlit|cloudflared" | grep -v grep

In [ ]:
import os

print("Current directory:", os.getcwd())
print()
print("Contents of /content:")
print(os.listdir("/content"))
print()
print("Does /content/drive exist?:", os.path.exists("/content/drive"))
print("Does /content/drive/MyDrive exist?:", os.path.exists("/content/drive/MyDrive"))

Current directory: /content/drive/MyDrive/NHL_Analytics_Hub

Contents of /content:
['.config', 'streamlit.log', 'drive', 'sample_data']

Does /content/drive exist?: True
Does /content/drive/MyDrive exist?: True


In [ ]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub
!pip install -r requirements.txt

/content/drive/MyDrive/NHL_Analytics_Hub
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 64.6 MB/s eta 0:00:00


In [ ]:
!streamlit --version

Streamlit, version 1.64.0


In [ ]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub
!streamlit run app.py --server.headless true --server.port 8501 > streamlit.log 2>&1 &

/content/drive/MyDrive/NHL_Analytics_Hub


In [ ]:
!sleep 5
!cat streamlit.log



2026-09-23 04:48:23.237 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.10.230.186:8501



In [ ]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Wed, 23 Sep 2026 04:49:14 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7260
last-modified: Wed, 23 Sep 2026 04:45:37 GMT
etag: "6c95fee04adc1477604953043cfbb216"
cache-control: no-cache



In [ ]:
!ps aux | grep -E "streamlit|cloudflared" | grep -v grep

root        7425  3.9  0.5 527920 74352 ?        Sl   04:48   0:02 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.headless true --server.port 8501


In [ ]:
!ls -lh /content/cloudflared

ls: cannot access '/content/cloudflared': No such file or directory


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!ls -lh /content/cloudflared

-rwxr-xr-x 1 root root 38M Sep 11 13:43 /content/cloudflared


In [ ]:
!/content/cloudflared tunnel --url http://localhost:8501

2026-09-23T04:51:33Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-23T04:51:33Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-23T04:51:36Z INF +--------------------------------------------------------------------------------------------+
2026-09-23T04:51:36Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-23T04:51:36Z INF |  https://contests-athens-intervention-task.trycloudfla

In [ ]:
from pathlib import Path

app_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app.py")
text = app_path.read_text()

start = text.find('elif page == "SQL Query":')

print(text[start:start + 1800])

elif page == "SQL Query":
    st.title("SQL Query")
    st.caption("Run read-only SELECT queries against the NHL Analytics Hub database.")

    query_options = ["-- write your own --"] + list(PREBUILT.keys())

    choice = st.selectbox(
        "Start from a pre-built query (optional)",
        query_options,
    )

    if choice == "-- write your own --":
        default_sql = "SELECT * FROM teams LIMIT 10;"
    else:
        default_sql = PREBUILT[choice]

    query = st.text_area(
        "SQL query",
        value=default_sql,
        height=220,
    )

    if st.button("Run query", type="primary"):
        cleaned_query = query.strip()

        if not cleaned_query:
            st.error("Please enter a SQL query.")
        elif not cleaned_query.lower().startswith("select"):
            st.error("Only SELECT queries are allowed here.")
        else:
            result = run_query(cleaned_query)

            if not result.empty:
                st.success(f"{len(result)} rows retur

In [ ]:
from pathlib import Path
import shutil

app_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app.py")
backup_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app_backup_before_sql_fix.py")

# Create backup
shutil.copy2(app_path, backup_path)
print(f"Backup created: {backup_path}")

text = app_path.read_text()

old_block = '''        if not cleaned_query:
            st.error("Please enter a SQL query.")
        elif not cleaned_query.lower().startswith("select"):
            st.error("Only SELECT queries are allowed here.")
        else:
            result = run_query(cleaned_query)
'''

new_block = '''        if not cleaned_query:
            st.error("Please enter a SQL query.")
        else:
            normalized_query = cleaned_query.lower().lstrip()

            blocked_keywords = (
                "insert",
                "update",
                "delete",
                "drop",
                "alter",
                "create",
                "replace",
                "attach",
                "detach",
                "pragma",
            )

            is_read_only = (
                normalized_query.startswith("select")
                or normalized_query.startswith("with")
            )

            contains_write_operation = any(
                normalized_query.startswith(keyword)
                or f" {keyword} " in normalized_query
                or f"\\n{keyword} " in normalized_query
                for keyword in blocked_keywords
            )

            if not is_read_only or contains_write_operation:
                st.error("Only read-only SELECT queries are allowed here.")
            else:
                result = run_query(cleaned_query)

                if not result.empty:
                    st.success(f"{len(result)} rows returned")
                    st.dataframe(
                        result,
                        hide_index=True,
                        use_container_width=True,
                    )
                else:
                    st.warning(
                        "Query ran but returned no rows "
                        "(or an error was shown above)."
                    )
'''

if old_block not in text:
    raise RuntimeError(
        "Expected SQL validation block was not found. "
        "No changes were made."
    )

text = text.replace(old_block, new_block, 1)
app_path.write_text(text)

print("SQL Query validation patched successfully.")
print(f"Updated: {app_path}")

Backup created: /content/drive/MyDrive/NHL_Analytics_Hub/app_backup_before_sql_fix.py
SQL Query validation patched successfully.
Updated: /content/drive/MyDrive/NHL_Analytics_Hub/app.py


In [ ]:
from pathlib import Path

app_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app.py")
text = app_path.read_text()

start = text.find('elif page == "SQL Query":')
print(text[start:start + 2500])

elif page == "SQL Query":
    st.title("SQL Query")
    st.caption("Run read-only SELECT queries against the NHL Analytics Hub database.")

    query_options = ["-- write your own --"] + list(PREBUILT.keys())

    choice = st.selectbox(
        "Start from a pre-built query (optional)",
        query_options,
    )

    if choice == "-- write your own --":
        default_sql = "SELECT * FROM teams LIMIT 10;"
    else:
        default_sql = PREBUILT[choice]

    query = st.text_area(
        "SQL query",
        value=default_sql,
        height=220,
    )

    if st.button("Run query", type="primary"):
        cleaned_query = query.strip()

        if not cleaned_query:
            st.error("Please enter a SQL query.")
        else:
            normalized_query = cleaned_query.lower().lstrip()

            blocked_keywords = (
                "insert",
                "update",
                "delete",
                "drop",
                "alter",
                "create",
       

In [ ]:
!pkill -f "streamlit run app.py" || true

^C


In [ ]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!streamlit run app.py --server.headless true --server.port 8501 > streamlit.log 2>&1 &

/content/drive/MyDrive/NHL_Analytics_Hub


In [ ]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Wed, 23 Sep 2026 05:12:51 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7260
last-modified: Wed, 23 Sep 2026 04:45:37 GMT
etag: "6c95fee04adc1477604953043cfbb216"
cache-control: no-cache



In [ ]:
!tail -30 streamlit.log



2026-09-23 05:12:45.844 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.10.230.186:8501



In [ ]:
!/content/cloudflared tunnel --url http://localhost:8501

2026-09-23T05:13:26Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-23T05:13:26Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-23T05:13:29Z INF +--------------------------------------------------------------------------------------------+
2026-09-23T05:13:29Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-23T05:13:29Z INF |  https://isolation-considering-wish-coupons.trycloudfl

In [ ]:
from pathlib import Path
import shutil

app_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app.py")
backup_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app_backup_before_sql_duplicate_fix.py")

# Create backup
shutil.copy2(app_path, backup_path)
print(f"Backup created: {backup_path}")

text = app_path.read_text()

duplicate_block = '''
            if not result.empty:
                st.success(f"{len(result)} rows returned")
                st.dataframe(
                    result,
                    hide_index=True,
                    use_container_width=True,
                )
            else:
                st.warning(
                    "Query ran but returned no rows "
                    "(or an error was shown above)."
                )
'''

# Find occurrences
count = text.count(duplicate_block)
print(f"Matching result blocks found: {count}")

if count != 2:
    raise RuntimeError(
        f"Expected exactly 2 matching result blocks, but found {count}. "
        "No changes were made."
    )

# Remove only the second occurrence.
first_index = text.find(duplicate_block)
second_index = text.find(duplicate_block, first_index + len(duplicate_block))

text = (
    text[:second_index]
    + text[second_index + len(duplicate_block):]
)

app_path.write_text(text)

print("Duplicate SQL result-rendering block removed successfully.")
print(f"Updated: {app_path}")

Backup created: /content/drive/MyDrive/NHL_Analytics_Hub/app_backup_before_sql_duplicate_fix.py
Matching result blocks found: 0


RuntimeError: Expected exactly 2 matching result blocks, but found 0. No changes were made.

In [ ]:
from pathlib import Path

app_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app.py")
text = app_path.read_text()

start = text.find("# ================================================================== SQL QUERY")
print(text[start:])

# ================================================================== SQL QUERY
elif page == "SQL Query":
    st.title("SQL Query")
    st.caption("Run read-only SELECT queries against the NHL Analytics Hub database.")

    query_options = ["-- write your own --"] + list(PREBUILT.keys())

    choice = st.selectbox(
        "Start from a pre-built query (optional)",
        query_options,
    )

    if choice == "-- write your own --":
        default_sql = "SELECT * FROM teams LIMIT 10;"
    else:
        default_sql = PREBUILT[choice]

    query = st.text_area(
        "SQL query",
        value=default_sql,
        height=220,
    )

    if st.button("Run query", type="primary"):
        cleaned_query = query.strip()

        if not cleaned_query:
            st.error("Please enter a SQL query.")
        else:
            normalized_query = cleaned_query.lower().lstrip()

            blocked_keywords = (
                "insert",
                "update",
                "delete",
   

In [ ]:
from pathlib import Path
import shutil

app_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app.py")
backup_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app_backup_before_sql_duplicate_fix_v2.py")

# Backup current file
shutil.copy2(app_path, backup_path)
print(f"Backup created: {backup_path}")

text = app_path.read_text()

section_marker = "# ================================================================== SQL QUERY"

start = text.find(section_marker)

if start == -1:
    raise RuntimeError("SQL QUERY section marker was not found. No changes made.")

new_sql_section = '''# ================================================================== SQL QUERY
elif page == "SQL Query":
    st.title("SQL Query")
    st.caption("Run read-only SELECT queries against the NHL Analytics Hub database.")

    query_options = ["-- write your own --"] + list(PREBUILT.keys())

    choice = st.selectbox(
        "Start from a pre-built query (optional)",
        query_options,
    )

    if choice == "-- write your own --":
        default_sql = "SELECT * FROM teams LIMIT 10;"
    else:
        default_sql = PREBUILT[choice]

    query = st.text_area(
        "SQL query",
        value=default_sql,
        height=220,
    )

    if st.button("Run query", type="primary"):
        cleaned_query = query.strip()

        if not cleaned_query:
            st.error("Please enter a SQL query.")
        else:
            normalized_query = cleaned_query.lower().lstrip()

            blocked_keywords = (
                "insert",
                "update",
                "delete",
                "drop",
                "alter",
                "create",
                "replace",
                "attach",
                "detach",
                "pragma",
            )

            is_read_only = (
                normalized_query.startswith("select")
                or normalized_query.startswith("with")
            )

            contains_write_operation = any(
                normalized_query.startswith(keyword)
                or f" {keyword} " in normalized_query
                or f"\\n{keyword} " in normalized_query
                for keyword in blocked_keywords
            )

            if not is_read_only or contains_write_operation:
                st.error("Only read-only SELECT queries are allowed here.")
            else:
                result = run_query(cleaned_query)

                if not result.empty:
                    st.success(f"{len(result)} rows returned")
                    st.dataframe(
                        result,
                        hide_index=True,
                        use_container_width=True,
                    )
                else:
                    st.warning(
                        "Query ran but returned no rows "
                        "(or an error was shown above)."
                    )
'''

# Replace everything from the SQL Query section to the end of the file.
text = text[:start] + new_sql_section

app_path.write_text(text)

print("SQL Query section replaced successfully.")
print("Duplicate result rendering removed.")

Backup created: /content/drive/MyDrive/NHL_Analytics_Hub/app_backup_before_sql_duplicate_fix_v2.py
SQL Query section replaced successfully.
Duplicate result rendering removed.


In [ ]:
from pathlib import Path

app_path = Path("/content/drive/MyDrive/NHL_Analytics_Hub/app.py")
text = app_path.read_text()

start = text.find("# ================================================================== SQL QUERY")
print(text[start:])

# ================================================================== SQL QUERY
elif page == "SQL Query":
    st.title("SQL Query")
    st.caption("Run read-only SELECT queries against the NHL Analytics Hub database.")

    query_options = ["-- write your own --"] + list(PREBUILT.keys())

    choice = st.selectbox(
        "Start from a pre-built query (optional)",
        query_options,
    )

    if choice == "-- write your own --":
        default_sql = "SELECT * FROM teams LIMIT 10;"
    else:
        default_sql = PREBUILT[choice]

    query = st.text_area(
        "SQL query",
        value=default_sql,
        height=220,
    )

    if st.button("Run query", type="primary"):
        cleaned_query = query.strip()

        if not cleaned_query:
            st.error("Please enter a SQL query.")
        else:
            normalized_query = cleaned_query.lower().lstrip()

            blocked_keywords = (
                "insert",
                "update",
                "delete",
   

In [ ]:
!python -m py_compile /content/drive/MyDrive/NHL_Analytics_Hub/app.py

In [ ]:
!pkill -f "streamlit run app.py" || true

^C


In [ ]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!streamlit run app.py --server.headless true --server.port 8501 > streamlit.log 2>&1 &

/content/drive/MyDrive/NHL_Analytics_Hub


In [ ]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Wed, 23 Sep 2026 05:27:30 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7260
last-modified: Wed, 23 Sep 2026 04:45:37 GMT
etag: "6c95fee04adc1477604953043cfbb216"
cache-control: no-cache



In [ ]:
!tail -30 streamlit.log



2026-09-23 05:27:25.413 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.10.230.186:8501



In [ ]:
!/content/cloudflared tunnel --url http://localhost:8501

2026-09-23T05:27:52Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-23T05:27:52Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-23T05:27:55Z INF +--------------------------------------------------------------------------------------------+
2026-09-23T05:27:55Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-23T05:27:55Z INF |  https://nissan-moving-medical-salary.trycloudflare.co

In [ ]:
conn.close()


In [3]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!git status

/content/drive/MyDrive/NHL_Analytics_Hub
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [4]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!git init

/content/drive/MyDrive/NHL_Analytics_Hub
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/NHL_Analytics_Hub/.git/


In [5]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!git branch -m main
!git branch --show-current

/content/drive/MyDrive/NHL_Analytics_Hub
main


In [7]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!cat > .gitignore <<'EOF'
# Python
__pycache__/
*.py[cod]
*.pyo

# Jupyter
.ipynb_checkpoints/

# Virtual environments
.venv/
venv/
env/

# Environment / secrets
.env
.env.*
!.env.example

# SQLite database
data/*.db
data/*.sqlite
data/*.sqlite3

# Streamlit / runtime logs
*.log

# Temporary / backup application files
*_backup*.py

# Colab / runtime artifacts
*.tmp
*.temp

# OS files
.DS_Store
Thumbs.db
EOF

cat .gitignore

# Python
__pycache__/
*.py[cod]
*.pyo

# Jupyter
.ipynb_checkpoints/

# Virtual environments
.venv/
venv/
env/

# Environment / secrets
.env
.env.*
!.env.example

# SQLite database
data/*.db
data/*.sqlite
data/*.sqlite3

# Streamlit / runtime logs
*.log

# Temporary / backup application files
*_backup*.py

# Colab / runtime artifacts
*.tmp
*.temp

# OS files
.DS_Store
Thumbs.db


In [8]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!git status --short

/content/drive/MyDrive/NHL_Analytics_Hub
?? .gitignore
?? README.md
?? app.py
?? app/
?? nhl_api_utils.py
?? notebooks/
?? queries.sql
?? requirements.txt
?? schema.sql


In [10]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

print("===== Root queries.sql =====")
!wc -l queries.sql
!head -20 queries.sql

print("\n===== app/queries.sql =====")
!wc -l app/queries.sql
!head -20 app/queries.sql

/content/drive/MyDrive/NHL_Analytics_Hub
===== Root queries.sql =====
95 queries.sql
-- NHL Analytics Hub — SQL Analysis Queries
-- Same queries as notebooks/04_sql_analysis.ipynb, kept here as a
-- standalone reference / for the SQL Query dashboard page.

-- 1. Which team has scored the most total goals this season?
SELECT t.team_name, s.goals_for
FROM standings s JOIN teams t ON t.team_id = s.team_id
ORDER BY s.goals_for DESC;

-- 2. Top 5 point scorers across the league
SELECT p.first_name, p.last_name, t.team_name, ss.points
FROM skater_season_stats ss
JOIN players p ON p.player_id = ss.player_id
JOIN teams t ON t.team_id = ss.team_id
ORDER BY ss.points DESC
LIMIT 5;

-- 3. Players with more than 20 goals AND more than 30 assists
SELECT p.first_name, p.last_name, ss.goals, ss.assists
FROM skater_season_stats ss

===== app/queries.sql =====
478 app/queries.sql
-- NHL Analytics Hub — SQL Analysis Queries
-- Validated against the current NHL Analytics Hub SQLite database.
--
-- Purpos

In [11]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!rm queries.sql

print("Root-level queries.sql removed.")
print("\nCurrent Git status:")
!git status --short

/content/drive/MyDrive/NHL_Analytics_Hub
Root-level queries.sql removed.

Current Git status:
?? .gitignore
?? README.md
?? app.py
?? app/
?? nhl_api_utils.py
?? notebooks/
?? requirements.txt
?? schema.sql


In [12]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!git remote add origin https://github.com/arunvijayvs/nhl-analytics-platform.git
!git remote -v

/content/drive/MyDrive/NHL_Analytics_Hub
origin	https://github.com/arunvijayvs/nhl-analytics-platform.git (fetch)
origin	https://github.com/arunvijayvs/nhl-analytics-platform.git (push)


In [13]:
%cd /content/drive/MyDrive/NHL_Analytics_Hub

!git status --short

/content/drive/MyDrive/NHL_Analytics_Hub
?? .gitignore
?? README.md
?? app.py
?? app/
?? nhl_api_utils.py
?? notebooks/
?? requirements.txt
?? schema.sql
